# MScFE 642 — GWP3 (Deep Learning for Finance)

This notebook is the **single source of code** for the project. It runs the full pipeline step by step, shows plots inline, and logs training/inference progress.

It **does not** include code for generating the LaTeX report (that is intentionally omitted).

## Environment

Use the local virtual environment created under `../.venv/`.

From a terminal:

```
cd /Users/alberto/Documents/projects/GWP_1/MLFinance/gwp3
source .venv/bin/activate
```

In [ ]:
from __future__ import annotations

import logging
import math
import os
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("gwp3-notebook")

def make_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = make_device()
logger.info("device=%s", device.type)

base_dir = Path("..").resolve()
images_dir = base_dir / "report" / "images"
images_dir.mkdir(parents=True, exist_ok=True)

@dataclass(frozen=True)
class ProjectConfig:
    symbol: str = "SPY"
    start: str = "2018-01-01"
    end: str = "2026-01-01"
    max_observations: int = 2000
    lookback: int = 32
    horizon: int = 1
    train_frac_single_split: float = 0.7
    seed: int = 7
    epochs_mlp: int = 20
    epochs_lstm: int = 25
    epochs_cnn: int = 25
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-4
    trading_cost_bps: float = 5.0
    lstm_hidden: int = 64

cfg = ProjectConfig()

def set_determinism(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(False)

set_determinism(cfg.seed)
logger.info("config symbol=%s lookback=%d horizon=%d seed=%d", cfg.symbol, cfg.lookback, cfg.horizon, cfg.seed)

def timed(msg: str, **fields: Any) -> tuple[float, str]:
    if fields:
        payload = " ".join(f"{k}={v}" for k, v in fields.items())
        logger.info("%s %s", msg, payload)
    else:
        logger.info("%s", msg)
    return time.perf_counter(), msg

def timed_done(t0: float, msg: str, **fields: Any) -> None:
    elapsed_s = time.perf_counter() - t0
    if fields:
        payload = " ".join(f"{k}={v}" for k, v in fields.items())
        logger.info("%s done elapsed_s=%.3f %s", msg, elapsed_s, payload)
    else:
        logger.info("%s done elapsed_s=%.3f", msg, elapsed_s)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 50)

logger.info("images_dir=%s", str(images_dir))

## Step 1a — Data gathering + description

Downloads an auto-adjusted close price series for a single security (default: `SPY`). Keeps at most 2,000 observations and saves:

- `data_price.png`
- `data_returns.png`
- `data_return_hist.png`

In [ ]:
def trim_to_max_observations(df: pd.DataFrame, max_observations: int) -> pd.DataFrame:
    if len(df) <= max_observations:
        return df
    return df.iloc[-max_observations:].copy()

def download_prices(symbol: str, start: str, end: str) -> pd.DataFrame:
    t0, msg = timed("download prices", symbol=symbol, start=start, end=end)
    df = yf.download(symbol, start=start, end=end, auto_adjust=True, progress=False)
    if df.empty:
        raise RuntimeError(f"No data downloaded for symbol={symbol!r}")
    df = df.rename_axis("date").reset_index()
    df["date"] = pd.to_datetime(df["date"], utc=False)
    df = df.sort_values("date").set_index("date")
    keep_cols = [c for c in ["Close", "Volume"] if c in df.columns]
    df = df[keep_cols].copy().dropna()
    timed_done(t0, msg, rows=len(df), cols=len(df.columns))
    return df

def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    up = delta.clip(lower=0.0)
    down = (-delta).clip(lower=0.0)
    roll_up = up.ewm(alpha=1 / period, adjust=False).mean()
    roll_down = down.ewm(alpha=1 / period, adjust=False).mean()
    rs = roll_up / (roll_down.replace(0.0, np.nan))
    rsi = 100.0 - (100.0 / (1.0 + rs))
    return rsi

def compute_macd(close: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9) -> tuple[pd.Series, pd.Series]:
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False).mean()
    return macd, macd_signal

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    t0, msg = timed("build features", rows=len(df))
    close = df["Close"]
    ret = np.log(close).diff()
    macd, macd_sig = compute_macd(close)
    rsi14 = compute_rsi(close, period=14)

    feats = pd.DataFrame(index=df.index)
    feats["ret"] = ret
    feats["ret_mean_5"] = ret.rolling(5).mean()
    feats["ret_mean_10"] = ret.rolling(10).mean()
    feats["ret_mean_20"] = ret.rolling(20).mean()
    feats["ret_std_5"] = ret.rolling(5).std()
    feats["ret_std_10"] = ret.rolling(10).std()
    feats["ret_std_20"] = ret.rolling(20).std()
    feats["rsi_14"] = rsi14
    feats["macd"] = macd
    feats["macd_signal"] = macd_sig
    if "Volume" in df.columns:
        vol = df["Volume"].replace(0.0, np.nan)
        feats["log_volume"] = np.log(vol)
        feats["log_volume_z20"] = (feats["log_volume"] - feats["log_volume"].rolling(20).mean()) / feats["log_volume"].rolling(20).std()
    feats = feats.replace([np.inf, -np.inf], np.nan).dropna()
    timed_done(t0, msg, rows=len(feats), cols=len(feats.columns))
    return feats

def build_labels_from_returns(features: pd.DataFrame, horizon: int = 1) -> pd.DataFrame:
    t0, msg = timed("build labels", rows=len(features), horizon=horizon)
    ret = features["ret"]
    fwd_ret = ret.shift(-horizon)
    y_class = (fwd_ret > 0.0).astype(int)
    out = pd.DataFrame(index=features.index)
    out["fwd_ret"] = fwd_ret
    out["y"] = y_class
    out = out.dropna()
    timed_done(t0, msg, rows=len(out), pos_rate=float(out["y"].mean()) if len(out) else float("nan"))
    return out

def save_price_and_return_plots(prices: pd.DataFrame, features: pd.DataFrame, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(prices.index, prices["Close"])
    ax.set_title("Price (Close, auto-adjusted)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Price")
    fig.tight_layout()
    fig.savefig(out_dir / "data_price.png", dpi=200)
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(features.index, features["ret"])
    ax.set_title("Log Returns")
    ax.set_xlabel("Date")
    ax.set_ylabel("Log return")
    fig.tight_layout()
    fig.savefig(out_dir / "data_returns.png", dpi=200)
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(features["ret"].dropna().values, bins=60, density=True)
    ax.set_title("Return Distribution")
    ax.set_xlabel("Log return")
    ax.set_ylabel("Density")
    fig.tight_layout()
    fig.savefig(out_dir / "data_return_hist.png", dpi=200)
    plt.show()
    plt.close(fig)

prices = download_prices(cfg.symbol, start=cfg.start, end=cfg.end)
prices = trim_to_max_observations(prices, cfg.max_observations)
features = build_features(prices)
features = trim_to_max_observations(features, cfg.max_observations)
labels = build_labels_from_returns(features, horizon=cfg.horizon)
labels = labels.reindex(features.index).dropna()
prices = prices.reindex(features.index).dropna()

save_price_and_return_plots(prices, features, images_dir)
logger.info("observations=%d date_range=%s->%s", len(prices), prices.index.min().date(), prices.index.max().date())
prices.head()

## Step 1b–1d — Single split, leakage intentionally present

Runs a single chronological train/test split and trains 3 deep learning models:

- MLP
- LSTM
- CNN on GAF

Leakage is introduced by scaling features using statistics computed on the full dataset before splitting.

In [ ]:
def make_supervised_sequences(
    features: pd.DataFrame,
    labels: pd.DataFrame,
    lookback: int,
    *,
    start_idx: int,
    end_idx_exclusive: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    t0, msg = timed("make sequences", lookback=lookback, start=start_idx, end=end_idx_exclusive)
    idx = features.index
    if not labels.index.equals(idx):
        labels = labels.reindex(idx)
    valid = labels["y"].notna()
    valid_positions = np.where(valid.values)[0]
    valid_positions = valid_positions[(valid_positions >= start_idx) & (valid_positions < end_idx_exclusive)]

    xs: list[np.ndarray] = []
    ys: list[int] = []
    fwd_rets: list[float] = []
    for pos in valid_positions:
        lb_start = pos - lookback + 1
        if lb_start < start_idx:
            continue
        window = features.iloc[lb_start : pos + 1].to_numpy(dtype=np.float32)
        if window.shape[0] != lookback:
            continue
        xs.append(window)
        ys.append(int(labels.iloc[pos]["y"]))
        fwd_rets.append(float(labels.iloc[pos]["fwd_ret"]))
    if not xs:
        timed_done(t0, msg, n_samples=0)
        return (
            np.empty((0, lookback, features.shape[1]), dtype=np.float32),
            np.empty((0,), dtype=np.int64),
            np.empty((0,), dtype=np.float32),
        )
    x = np.stack(xs, axis=0)
    y = np.asarray(ys, dtype=np.int64)
    r = np.asarray(fwd_rets, dtype=np.float32)
    timed_done(t0, msg, n_samples=int(x.shape[0]), n_features=int(x.shape[2]))
    return x, y, r

def gasf_image(series: np.ndarray) -> np.ndarray:
    s = np.asarray(series, dtype=np.float32)
    s_min = float(np.min(s))
    s_max = float(np.max(s))
    denom = s_max - s_min
    if denom == 0.0:
        s_scaled = np.zeros_like(s, dtype=np.float32)
    else:
        s_scaled = 2.0 * ((s - s_min) / denom) - 1.0
        s_scaled = np.clip(s_scaled, -1.0, 1.0)
    phi = np.arccos(s_scaled)
    gaf = np.cos(phi[:, None] + phi[None, :]).astype(np.float32)
    return gaf

def gasf_image_with_global_minmax(series: np.ndarray, global_min: float, global_max: float) -> np.ndarray:
    s = np.asarray(series, dtype=np.float32)
    denom = global_max - global_min
    if denom == 0.0:
        s_scaled = np.zeros_like(s, dtype=np.float32)
    else:
        s_scaled = 2.0 * ((s - global_min) / denom) - 1.0
        s_scaled = np.clip(s_scaled, -1.0, 1.0)
    phi = np.arccos(s_scaled)
    gaf = np.cos(phi[:, None] + phi[None, :]).astype(np.float32)
    return gaf

class MLP(nn.Module):
    def __init__(self, in_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

class LSTMClassifier(nn.Module):
    def __init__(self, n_features: int, hidden: int = 64) -> None:
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden, num_layers=1, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)

class GAFConvNet(nn.Module):
    def __init__(self, image_size: int) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        h = image_size // 4
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32 * h * h, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.features(x)
        return self.head(z).squeeze(-1)

def split_train_val_chrono(x: np.ndarray, y: np.ndarray, val_frac: float = 0.2) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n = x.shape[0]
    cut = max(1, int(math.floor(n * (1.0 - val_frac))))
    x_train = x[:cut]
    y_train = y[:cut]
    x_val = x[cut:]
    y_val = y[cut:]
    return x_train, y_train, x_val, y_val

def to_torch(x: np.ndarray) -> torch.Tensor:
    return torch.tensor(x, dtype=torch.float32, device=device)

def train_binary_classifier(
    model: nn.Module,
    x_train: torch.Tensor,
    y_train: torch.Tensor,
    x_val: torch.Tensor,
    y_val: torch.Tensor,
    *,
    epochs: int,
    batch_size: int,
    lr: float,
    weight_decay: float,
) -> nn.Module:
    t0 = time.perf_counter()
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    best_state: dict[str, Any] | None = None
    best_val = float("inf")
    patience = 5
    bad = 0

    n = x_train.shape[0]
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n, device=device)
        train_loss_sum = 0.0
        train_count = 0
        for i in range(0, n, batch_size):
            idx = perm[i : i + batch_size]
            xb = x_train[idx]
            yb = y_train[idx].float()
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            train_loss_sum += float(loss.item()) * int(xb.shape[0])
            train_count += int(xb.shape[0])

        model.eval()
        with torch.no_grad():
            val_logits = model(x_val)
            val_loss = float(loss_fn(val_logits, y_val.float()).item())

        train_loss = train_loss_sum / max(1, train_count)
        logger.info(
            "epoch=%d/%d train_loss=%.6f val_loss=%.6f",
            int(epoch),
            int(epochs),
            float(train_loss),
            float(val_loss),
        )
        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                logger.info("early stop patience=%d", patience)
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    elapsed_s = time.perf_counter() - t0
    logger.info("train done arch=%s n_train=%d n_val=%d best_val=%.6f elapsed_s=%.3f", model.__class__.__name__, int(x_train.shape[0]), int(x_val.shape[0]), float(best_val), float(elapsed_s))
    return model

def predict_proba(model: nn.Module, x: torch.Tensor) -> np.ndarray:
    t0 = time.perf_counter()
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        logits = model(x).detach().cpu().numpy()
    proba = 1.0 / (1.0 + np.exp(-logits))
    elapsed_s = time.perf_counter() - t0
    logger.info("predict done arch=%s n=%d elapsed_s=%.3f", model.__class__.__name__, int(proba.shape[0]), float(elapsed_s))
    return proba.astype(np.float64)

def evaluate_predictions(y_true: np.ndarray, proba: np.ndarray) -> dict[str, float]:
    y_pred = (proba >= 0.5).astype(int)
    out: dict[str, float] = {"accuracy": float(accuracy_score(y_true, y_pred))}
    if len(np.unique(y_true)) > 1:
        out["roc_auc"] = float(roc_auc_score(y_true, proba))
    else:
        out["roc_auc"] = float("nan")
    return out

def compute_strategy_returns_from_proba(fwd_returns: np.ndarray, proba_up: np.ndarray, trading_cost_bps: float) -> pd.Series:
    pos = np.where(proba_up >= 0.5, 1.0, -1.0)
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    turnover = np.abs(pos - pos_prev)
    cost = (trading_cost_bps / 1e4) * turnover
    strat = pos * fwd_returns - cost
    return pd.Series(strat)

def equity_curve(returns: pd.Series) -> pd.Series:
    return (1.0 + returns.fillna(0.0)).cumprod()

def max_drawdown(equity: pd.Series) -> float:
    peak = equity.cummax()
    dd = equity / peak - 1.0
    return float(dd.min())

def annualized_sharpe(daily_returns: pd.Series, periods_per_year: int = 252) -> float:
    r = daily_returns.dropna().to_numpy(dtype=float)
    if r.size < 2:
        return float("nan")
    mu = r.mean()
    sd = r.std(ddof=1)
    if sd == 0.0:
        return float("nan")
    return float((mu / sd) * math.sqrt(periods_per_year))

def summarize_backtest(daily_returns: pd.Series) -> dict[str, float]:
    eq = equity_curve(daily_returns)
    total = float(eq.iloc[-1] - 1.0) if len(eq) else float("nan")
    mdd = max_drawdown(eq) if len(eq) else float("nan")
    sharpe = annualized_sharpe(daily_returns)
    hit = float((daily_returns > 0).mean()) if len(daily_returns) else float("nan")
    return {"total_return": total, "max_drawdown": mdd, "sharpe": sharpe, "hit_rate": hit}

def save_equity_plot(daily_returns: pd.Series, title: str, out_path: Path) -> None:
    eq = equity_curve(daily_returns)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(eq.index, eq.values)
    ax.set_title(title)
    ax.set_xlabel("Index")
    ax.set_ylabel("Equity (start=1.0)")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.show()
    plt.close(fig)

def single_split_indices(n: int, train_frac: float) -> tuple[int, int]:
    train_end = max(1, int(math.floor(n * train_frac)))
    return 0, train_end

logger.info("step1 start")
n = len(features)
start, train_end = single_split_indices(n, cfg.train_frac_single_split)

scaler = StandardScaler()
scaled_all = pd.DataFrame(scaler.fit_transform(features.values), index=features.index, columns=features.columns)

x_train, y_train, r_train = make_supervised_sequences(scaled_all, labels, cfg.lookback, start_idx=start, end_idx_exclusive=train_end)
x_test, y_test, r_test = make_supervised_sequences(scaled_all, labels, cfg.lookback, start_idx=train_end, end_idx_exclusive=n)
logger.info("step1 split n=%d train_end=%d x_train=%d x_test=%d", int(n), int(train_end), int(x_train.shape[0]), int(x_test.shape[0]))

ret_series = features["ret"].to_numpy(dtype=np.float32)
global_min = float(np.min(ret_series))
global_max = float(np.max(ret_series))
ret_col = list(features.columns).index("ret")
w_train = x_train[:, :, ret_col]
w_test = x_test[:, :, ret_col]

t0, msg = timed("train+infer step1", n_train=int(x_train.shape[0]), n_test=int(x_test.shape[0]))

# MLP
x_train_f = x_train.reshape(x_train.shape[0], -1)
x_test_f = x_test.reshape(x_test.shape[0], -1)
x_tr, y_tr, x_val, y_val = split_train_val_chrono(x_train_f, y_train)
logger.info("MLP")
mlp = MLP(in_dim=x_tr.shape[1])
mlp = train_binary_classifier(mlp, to_torch(x_tr), torch.tensor(y_tr, dtype=torch.int64, device=device), to_torch(x_val), torch.tensor(y_val, dtype=torch.int64, device=device), epochs=cfg.epochs_mlp, batch_size=cfg.batch_size, lr=cfg.lr, weight_decay=cfg.weight_decay)
proba_mlp = predict_proba(mlp, to_torch(x_test_f))

# LSTM
x_tr, y_tr, x_val, y_val = split_train_val_chrono(x_train, y_train)
logger.info("LSTM")
lstm = LSTMClassifier(n_features=x_tr.shape[2], hidden=cfg.lstm_hidden)
lstm = train_binary_classifier(lstm, to_torch(x_tr), torch.tensor(y_tr, dtype=torch.int64, device=device), to_torch(x_val), torch.tensor(y_val, dtype=torch.int64, device=device), epochs=cfg.epochs_lstm, batch_size=cfg.batch_size, lr=cfg.lr, weight_decay=cfg.weight_decay)
proba_lstm = predict_proba(lstm, to_torch(x_test))

# CNN on GAF
def make_gaf_images(windows: np.ndarray, *, gmin: float, gmax: float) -> np.ndarray:
    imgs = [gasf_image_with_global_minmax(w, global_min=gmin, global_max=gmax) for w in windows]
    x = np.stack(imgs, axis=0)
    return x[:, None, :, :]

x_train_img = make_gaf_images(w_train, gmin=global_min, gmax=global_max)
x_test_img = make_gaf_images(w_test, gmin=global_min, gmax=global_max)
x_tr, y_tr, x_val, y_val = split_train_val_chrono(x_train_img, y_train)
logger.info("CNN+GAF")
cnn = GAFConvNet(image_size=x_tr.shape[-1])
cnn = train_binary_classifier(cnn, to_torch(x_tr), torch.tensor(y_tr, dtype=torch.int64, device=device), to_torch(x_val), torch.tensor(y_val, dtype=torch.int64, device=device), epochs=cfg.epochs_cnn, batch_size=cfg.batch_size, lr=cfg.lr, weight_decay=cfg.weight_decay)
proba_cnn = predict_proba(cnn, to_torch(x_test_img))

timed_done(t0, msg)

rows = []
for name, proba in [("mlp", proba_mlp), ("lstm", proba_lstm), ("cnn_gaf", proba_cnn)]:
    metrics = evaluate_predictions(y_test, proba)
    strat_rets = compute_strategy_returns_from_proba(r_test, proba, cfg.trading_cost_bps)
    summary = summarize_backtest(strat_rets)
    save_equity_plot(strat_rets, f"Step 1 Equity Curve ({name})", images_dir / f"step1_equity_{name}.png")
    rows.append({"step": "1", "model": name, **metrics, **summary})

step1 = pd.DataFrame(rows)
logger.info("step1 done")
step1

## Step 2 — Walk-forward backtests (leakage present)

Non-anchored walk-forward evaluation with:

- Step 2a: 500/500 train/test
- Step 2b: 500/100 train/test

Leakage remains (global scaling).

In [ ]:
def walk_forward_splits(n: int, train_size: int, test_size: int) -> list[tuple[int, int, int, int]]:
    out = []
    start = 0
    while True:
        train_start = start
        train_end = train_start + train_size
        test_start = train_end
        test_end = test_start + test_size
        if test_end > n:
            break
        out.append((train_start, train_end, test_start, test_end))
        start = test_start
    return out

def wf_run(*, tag: str, train_size: int, test_size: int, leakage_mode: Literal["leaky", "reduced"]) -> pd.DataFrame:
    t0, msg = timed("walk-forward", tag=tag, train_size=train_size, test_size=test_size, leakage=leakage_mode)
    n = len(features)
    splits = walk_forward_splits(n, train_size=train_size, test_size=test_size)
    logger.info("wf splits tag=%s n=%d n_splits=%d", tag, int(n), int(len(splits)))

    if leakage_mode == "leaky":
        scaler = StandardScaler()
        scaled_features = pd.DataFrame(scaler.fit_transform(features.values), index=features.index, columns=features.columns)
        ret_series = features["ret"].to_numpy(dtype=np.float32)
        w_global_min = float(np.min(ret_series))
        w_global_max = float(np.max(ret_series))

    model_returns: dict[str, list[pd.Series]] = {"mlp": [], "lstm": [], "cnn_gaf": []}
    model_y_true: dict[str, list[np.ndarray]] = {"mlp": [], "lstm": [], "cnn_gaf": []}
    model_proba: dict[str, list[np.ndarray]] = {"mlp": [], "lstm": [], "cnn_gaf": []}
    ret_col = list(features.columns).index("ret")
    used_folds = 0

    for fold_idx, (train_start, train_end, test_start, test_end) in enumerate(splits, start=1):
        fold_t0 = time.perf_counter()
        if leakage_mode == "leaky":
            feat_fold = scaled_features
            test_start_seq = test_start
            gmin = w_global_min
            gmax = w_global_max
        else:
            scaler = StandardScaler()
            scaler.fit(features.iloc[train_start:train_end].values)
            feat_fold = pd.DataFrame(scaler.transform(features.values), index=features.index, columns=features.columns)
            train_end = max(train_start, train_end - cfg.horizon)
            test_start_seq = test_start + cfg.lookback
            if test_start_seq >= test_end:
                continue
            ret_train = features.iloc[train_start:train_end]["ret"].to_numpy(dtype=np.float32)
            gmin = float(np.min(ret_train))
            gmax = float(np.max(ret_train))

        x_train, y_train, _ = make_supervised_sequences(feat_fold, labels, cfg.lookback, start_idx=train_start, end_idx_exclusive=train_end)
        x_test, y_test, r_test = make_supervised_sequences(feat_fold, labels, cfg.lookback, start_idx=test_start_seq, end_idx_exclusive=test_end)
        if x_train.shape[0] < 50 or x_test.shape[0] < 10:
            logger.info("wf skip tag=%s fold=%d x_train=%d x_test=%d", tag, int(fold_idx), int(x_train.shape[0]), int(x_test.shape[0]))
            continue
        logger.info("wf fold start tag=%s fold=%d x_train=%d x_test=%d", tag, int(fold_idx), int(x_train.shape[0]), int(x_test.shape[0]))

        # Train + predict (fold)
        # MLP
        x_train_f = x_train.reshape(x_train.shape[0], -1)
        x_test_f = x_test.reshape(x_test.shape[0], -1)
        x_tr, y_tr, x_val, y_val = split_train_val_chrono(x_train_f, y_train)
        mlp = MLP(in_dim=x_tr.shape[1])
        mlp = train_binary_classifier(mlp, to_torch(x_tr), torch.tensor(y_tr, dtype=torch.int64, device=device), to_torch(x_val), torch.tensor(y_val, dtype=torch.int64, device=device), epochs=cfg.epochs_mlp, batch_size=cfg.batch_size, lr=cfg.lr, weight_decay=cfg.weight_decay)
        proba_mlp = predict_proba(mlp, to_torch(x_test_f))

        # LSTM
        x_tr, y_tr, x_val, y_val = split_train_val_chrono(x_train, y_train)
        lstm = LSTMClassifier(n_features=x_tr.shape[2], hidden=cfg.lstm_hidden)
        lstm = train_binary_classifier(lstm, to_torch(x_tr), torch.tensor(y_tr, dtype=torch.int64, device=device), to_torch(x_val), torch.tensor(y_val, dtype=torch.int64, device=device), epochs=cfg.epochs_lstm, batch_size=cfg.batch_size, lr=cfg.lr, weight_decay=cfg.weight_decay)
        proba_lstm = predict_proba(lstm, to_torch(x_test))

        # CNN+GAF
        w_train = x_train[:, :, ret_col]
        w_test = x_test[:, :, ret_col]
        x_train_img = np.stack([gasf_image_with_global_minmax(w, global_min=gmin, global_max=gmax) for w in w_train], axis=0)[:, None, :, :]
        x_test_img = np.stack([gasf_image_with_global_minmax(w, global_min=gmin, global_max=gmax) for w in w_test], axis=0)[:, None, :, :]
        x_tr, y_tr, x_val, y_val = split_train_val_chrono(x_train_img, y_train)
        cnn = GAFConvNet(image_size=x_tr.shape[-1])
        cnn = train_binary_classifier(cnn, to_torch(x_tr), torch.tensor(y_tr, dtype=torch.int64, device=device), to_torch(x_val), torch.tensor(y_val, dtype=torch.int64, device=device), epochs=cfg.epochs_cnn, batch_size=cfg.batch_size, lr=cfg.lr, weight_decay=cfg.weight_decay)
        proba_cnn = predict_proba(cnn, to_torch(x_test_img))

        for name, proba in [("mlp", proba_mlp), ("lstm", proba_lstm), ("cnn_gaf", proba_cnn)]:
            strat_rets = compute_strategy_returns_from_proba(r_test, proba, cfg.trading_cost_bps)
            model_returns[name].append(strat_rets)
            model_y_true[name].append(y_test)
            model_proba[name].append(proba)

        used_folds += 1
        logger.info("wf fold done tag=%s fold=%d elapsed_s=%.3f", tag, int(fold_idx), float(time.perf_counter() - fold_t0))

    rows = []
    for name in ["mlp", "lstm", "cnn_gaf"]:
        if not model_returns[name]:
            continue
        all_rets = pd.concat(model_returns[name], ignore_index=True)
        all_y = np.concatenate(model_y_true[name], axis=0)
        all_p = np.concatenate(model_proba[name], axis=0)
        metrics = evaluate_predictions(all_y, all_p)
        summary = summarize_backtest(all_rets)
        save_equity_plot(all_rets, f"Step {tag} Equity Curve ({name})", images_dir / f"step{tag}_equity_{name}.png")
        rows.append({"step": tag, "model": name, **metrics, **summary})

    timed_done(t0, msg, used_folds=int(used_folds))
    return pd.DataFrame(rows)

logger.info("step2 start")
step2a = wf_run(tag="2a", train_size=500, test_size=500, leakage_mode="leaky")
step2b = wf_run(tag="2b", train_size=500, test_size=100, leakage_mode="leaky")
step2 = pd.concat([step2a, step2b], ignore_index=True)
logger.info("step2 done")
step2

## Step 3 — Walk-forward backtests (leakage reduced)

Leakage mitigation:

- scalers are fit only on each fold's training window
- test windows are made stricter by skipping the first `lookback` observations of each test window
- the last `horizon` observations of each training window are dropped (purge)
- GAF normalization uses training-only return min/max per fold

In [ ]:
logger.info("step3 start")
step3b = wf_run(tag="3b", train_size=500, test_size=500, leakage_mode="reduced")
step3c = wf_run(tag="3c", train_size=500, test_size=100, leakage_mode="reduced")
step3 = pd.concat([step3b, step3c], ignore_index=True)
logger.info("step3 done")
step3

## Combined summary

Combine Step 1–3 metrics for comparison. (No LaTeX/report generation is performed here.)

In [ ]:
summary = pd.concat([step1, step2, step3], ignore_index=True)
summary.to_csv(images_dir / "metrics_summary.csv", index=False)
logger.info("saved metrics_summary.csv")
summary

## Saved artifacts

The pipeline saves (in `../report/images/`):

- Figures (`.png`) for prices, returns, and equity curves
- `metrics_summary.csv`

This notebook intentionally omits report generation.

In [ ]:
sorted([p.name for p in images_dir.glob("*.png")])[:20], (images_dir / "metrics_summary.csv").exists()